# 02 — Exploratory Data Analysis (EDA)

Explore sales trends, product families, store performance, weekday patterns, and promotions. These analyses help us understand the business before building forecasting models.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path("/content/drive/MyDrive/retail-demand-forecasting")
DATA_DIR = PROJECT / "data/raw"
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"])
print("Dataset shape:", train.shape)
display(train.head())

## 1. Overall sales trend

Aggregate sales by day, then examine monthly totals.

In [ ]:
daily_sales = train.groupby("date", as_index=False)["sales"].sum()
daily_sales["month"] = daily_sales["date"].dt.to_period("M").dt.to_timestamp()
monthly_sales = daily_sales.groupby("month", as_index=False)["sales"].sum()

plt.figure(figsize=(13, 5))
plt.plot(monthly_sales["month"], monthly_sales["sales"])
plt.title("Monthly Total Sales")
plt.xlabel("Month")
plt.ylabel("Total Sales")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

display(monthly_sales.tail(12))

## 2. Product-family performance

Identify which product families contribute the most total sales.

In [ ]:
family_sales = train.groupby("family")["sales"].sum().sort_values(ascending=False)
plt.figure(figsize=(10, 7))
family_sales.head(15).sort_values().plot(kind="barh")
plt.title("Top 15 Product Families by Total Sales")
plt.xlabel("Total Sales")
plt.tight_layout()
plt.show()

display(family_sales.head(15).to_frame("total_sales"))

## 3. Store performance

Compare total sales across stores. Total sales alone do not indicate profitability.

In [ ]:
store_sales = train.groupby("store_nbr")["sales"].sum().sort_values(ascending=False)
plt.figure(figsize=(11, 5))
store_sales.head(10).sort_values().plot(kind="barh")
plt.title("Top 10 Stores by Total Sales")
plt.xlabel("Total Sales")
plt.tight_layout()
plt.show()

display(store_sales.head(10).to_frame("total_sales"))

## 4. Weekday patterns

Compare average daily sales by day of the week.

In [ ]:
daily_sales["weekday"] = daily_sales["date"].dt.day_name()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_sales = daily_sales.groupby("weekday")["sales"].mean().reindex(weekday_order)
weekday_sales.plot(kind="bar", figsize=(10, 4))
plt.title("Average Daily Sales by Weekday")
plt.xlabel("Day")
plt.ylabel("Average Total Sales")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

display(weekday_sales.to_frame("average_daily_sales"))

## 5. Promotion comparison

Compare records with and without promotions. This is descriptive analysis and does not establish that promotions cause increased sales.

In [ ]:
train["promotion_status"] = np.where(train["onpromotion"].fillna(0) > 0, "Promotion present", "No promotion")
promotion_summary = train.groupby("promotion_status").agg(
    row_count=("sales", "size"),
    average_sales=("sales", "mean"),
    median_sales=("sales", "median"),
    average_promotion_count=("onpromotion", "mean")
)
display(promotion_summary)